# torchrolling on a GPU

Checks that torchrolling is correct on a real GPU and benchmarks it against pandas, polars,
cuDF and a plain `torch.unfold`.

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100), and Settings →
**Internet on**. Then Run All. The whole notebook takes about 20–40 minutes.

In [ ]:
!nvidia-smi

## Install

Clones the repository and installs it on top of Kaggle's own torch (which already has CUDA
and Triton). If the repository is not on GitHub yet, upload the project folder as a Kaggle
dataset instead and set `SRC` to its path, e.g. `/kaggle/input/torchrolling`.

In [ ]:
REPO = "https://github.com/LenaBarretta/torchrolling"
SRC = "/kaggle/working/torchrolling"

import os
if not os.path.exists(SRC):
    !git clone -q {REPO} {SRC}
!pip install -q {SRC} pytest pytest-cov hypothesis mpmath pandas polars

In [ ]:
import torch, triton, torchrolling
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| triton", triton.__version__)
print("torchrolling", torchrolling.__version__, "|", torch.cuda.get_device_name(0))

Optional: cuDF (RAPIDS), the closest GPU competitor. The install is large and can take a few
minutes; skip this cell to benchmark without it.

In [ ]:
!pip install -q --extra-index-url=https://pypi.nvidia.com cudf-cu12

## Correctness on the GPU

The full test suite. With CUDA available, the `test_cuda_*` tests also run: they compare the
GPU results (including the Triton quantile kernel) with the CPU code path, which itself is
checked against pandas.

In [ ]:
!cd {SRC} && python -m pytest -q -p no:cacheprovider --no-cov tests

## Benchmark

Methods: rolling mean, std, max, median, corr, and an EWM mean. On CUDA torchrolling runs
twice: with its default float64 accumulator and with `acc_dtype=torch.float32` (the
`f32` column), which shows what float64 costs on this GPU. The largest case is 10,000
series × 10,000 points; CPU libraries are skipped above 10 million values
(`--max-cpu-elements`).

In [ ]:
!cd {SRC} && python bench/bench.py

In [ ]:
from pathlib import Path
from IPython.display import Markdown

results = sorted(Path(SRC, "bench", "results").glob("*.md"))
Markdown("\n\n".join(p.read_text() for p in results))

## Keep the results

Download `bench-results.zip` (Output panel on the right, after the cell below), unpack it
into the repository so that `bench/results/` holds the `.json` and `.md` files, and send
them over: the README table is generated from them.

In [ ]:
!cd {SRC} && zip -qr /kaggle/working/bench-results.zip bench/results && ls -l /kaggle/working/bench-results.zip